# 03.4 — Machine Learning Tweedie Regression Benchmark

Huấn luyện và đánh giá mô hình hồi quy phân phối Tweedie deviance loss trực tiếp trên chuỗi lượng mưa có lạm phát số 0.


In [1]:
import sys
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Tự động tìm project root từ bất kỳ thư mục con nào
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.training.benchmark_utils import (
    load_canonical_data,
    compute_benchmark_metrics,
    save_benchmark_artifacts,
    build_purged_direct_samples,
    UNIFIED_SCHEMA,
)

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from src.featurengineering.builder import FeatureBuilder, FeatureConfig

train_full, val_set, test_set = load_canonical_data()


## 2. Chuẩn bị Dữ liệu & Purged Multi-Horizon Samples

In [2]:
fb_config = FeatureConfig(
    lag_columns=["Lượng mưa", "Nhiệt độ 2m", "Độ ẩm tương đối 2m", "Tốc độ gió 2m"],
    lag_periods=[1, 2, 3, 7],
    rolling_columns=["Lượng mưa", "Nhiệt độ 2m"],
    rolling_windows=[7, 14, 30],
    rolling_stats=["mean", "std"],
    temporal_features=["Month_sin", "Month_cos", "DayOfYear_sin", "DayOfYear_cos", "Is_Wet_Season"],
)

fb = FeatureBuilder(fb_config)
fb.fit(train_full)

train_feat = fb.transform(train_full)
test_feat = fb.transform(test_set)
feature_cols = [c for c in train_feat.columns if c not in ['Ngày', 'Lượng mưa']]

horizon = 7
X_train, Y_train, _ = build_purged_direct_samples(train_feat, feature_cols, horizon=horizon)
X_test, Y_test, _ = build_purged_direct_samples(test_feat, feature_cols, horizon=horizon)

sub_indices = np.linspace(0, len(X_test) - 1, min(50, len(X_test)), dtype=int)
X_test_eval = X_test[sub_indices]
Y_test_eval = Y_test[sub_indices]
print(f"Purged training samples: {X_train.shape[0]}, Test evaluation samples: {X_test_eval.shape[0]}")


FeatureBuilder.fit() - learning from train data only
[STATS] fit_missing_value_stats: computed fill_mean for 30 columns (train only)
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 2m', 'Độ ẩm tương đối 2m', 'Tốc độ gió 2m']
   Lag periods: [1, 2, 3, 7]
   Dataset shape: (7426, 47)
[ROLLING] CREATING ROLLING FEATURES
   [LEAK-SAFE] Leakage-safe mode: shift(1) applied before rolling
   [OK] Created 12 rolling features
   Columns: ['Lượng mưa', 'Nhiệt độ 2m']
   Windows: [7, 14, 30]
   Statistics: ['mean', 'std']
   Dataset shape: (7426, 59)
[TEMPORAL] CREATING TEMPORAL FEATURES
   [OK] Created 5 temporal features
   Dataset shape: (7426, 64)
[STATS] fit_feature_quality: 62 features analyzed (train only)
   Keep: 55, Drop: 7
   FeatureBuilder fitted
[FEATURE-APPLY] apply_missing_value_fill: filled 0 values using train-derived stats
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 

## 3. Huấn luyện & Đánh giá Mô hình Tweedie

In [3]:
tweedie_models = {
    'LightGBM_Tweedie': MultiOutputRegressor(LGBMRegressor(
        objective='tweedie',
        tweedie_variance_power=1.5,
        n_estimators=100,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        verbosity=-1,
        random_state=42
    )),
    'XGBoost_Tweedie': MultiOutputRegressor(XGBRegressor(
        objective='reg:tweedie',
        tweedie_variance_power=1.5,
        n_estimators=100,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        verbosity=0,
        random_state=42
    ))
}

results_list = []
forecasts_dict = {}

for name, model in tweedie_models.items():
    print(f"Training {name}...")
    t0 = time.time()
    model.fit(X_train, Y_train)
    t_train = time.time() - t0
    
    t0 = time.time()
    y_pred = model.predict(X_test_eval)
    t_infer = time.time() - t0
    
    y_pred = np.clip(y_pred, 0, None)
    forecasts_dict[name] = y_pred
    
    metrics = compute_benchmark_metrics(
        y_true=Y_test_eval,
        y_pred=y_pred,
        model_name=name,
        group='Tweedie Single-Stage',
        strategy='Compound Poisson-Gamma (p=1.5)',
        training_time=t_train,
        inference_time=t_infer,
        parameter_count=600,
        rain_threshold=0.1
    )
    results_list.append(metrics)

df_results = pd.concat(results_list, ignore_index=True).sort_values(by='CR-MAE@7')
csv_path, npz_path = save_benchmark_artifacts(df_results, forecasts_dict, Y_test_eval, "ml_tweedie")
print(f">> Saved artifacts to:\n  - {csv_path}\n  - {npz_path}\n")

print("=== ML TWEEDIE BENCHMARK RESULTS (Ranked by CR-MAE@7) ===")
display(df_results[['model', 'strategy', 'MAE', 'RMSE', 'rain_day_MAE', 'CR-MAE@7', 'CR-Bias@7']])


Training LightGBM_Tweedie...
Training XGBoost_Tweedie...
>> Saved artifacts to:
  - data\processed\results\ml_tweedie_results.csv
  - data\processed\results\ml_tweedie_forecasts.npz

=== ML TWEEDIE BENCHMARK RESULTS (Ranked by CR-MAE@7) ===
              model                        strategy  ...  CR-MAE@7  CR-Bias@7
0  LightGBM_Tweedie  Compound Poisson-Gamma (p=1.5)  ...   17.3842    -9.1059
1   XGBoost_Tweedie  Compound Poisson-Gamma (p=1.5)  ...   17.7567    -8.9689

[2 rows x 7 columns]
